# Notebook 02 — Sentiment Extraction from RBI Documents
**Project:** RBI Communication, Financial Stability and Macroeconomic Transmission:
Evidence from Sentiment Analysis of MPC Minutes (2010–2025)

---
### How this notebook works — 3-layer approach

| Layer | When it runs | What it does |
|-------|-------------|--------------|
| **Layer 1** | Always first | Download PDFs directly from RBI website |
| **Layer 2** | If Layer 1 blocked | Read PDFs from a local folder you saved manually |
| **Layer 3** | If both fail | Convert your existing scraped CSV data (from old project) |

> **Note on 403 errors:** RBI blocks automated downloads from cloud/server IPs.
> On your own laptop this usually works fine. If it still blocks, use Layer 2 or Layer 3.

## Step 0 — Install libraries

In [21]:
import subprocess
for lib in ['pypdf','pdfplumber','PyPDF2','pysentiment2','pandas','numpy','requests','openpyxl']:
    r = subprocess.run(['pip','install',lib,'-q','--break-system-packages'],
                       capture_output=True, text=True)
    print(f"  {'OK' if r.returncode==0 else 'FAIL'}  {lib}")
print("\nDone.")

  OK  pypdf
  OK  pdfplumber
  OK  PyPDF2
  OK  pysentiment2
  OK  pandas
  OK  numpy
  OK  requests
  OK  openpyxl

Done.


## Step 1 — Imports and project paths

In [22]:
import pandas as pd
import numpy as np
import os, requests, time, zipfile
import warnings
warnings.filterwarnings('ignore')
from io import BytesIO

import PyPDF2
try:
    from pypdf import PdfReader as pypdf_Reader
    PYPDF_AVAIL = True
except ImportError:
    PYPDF_AVAIL = False

try:
    import pdfplumber
    PDFPLUMBER_AVAIL = True
except ImportError:
    PDFPLUMBER_AVAIL = False

import pysentiment2 as ps
lm_model = ps.LM()

# ── Paths ─────────────────────────────────────────────────────────────────────
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA_RAW     = os.path.join(PROJECT_ROOT, 'data', 'raw')
DATA_PROC    = os.path.join(PROJECT_ROOT, 'data', 'processed')
# Folder where you can manually save PDFs (Layer 2 fallback)
LOCAL_PDF_DIR = os.path.join(PROJECT_ROOT, 'data', 'raw', 'pdfs')
os.makedirs(DATA_PROC,    exist_ok=True)
os.makedirs(LOCAL_PDF_DIR, exist_ok=True)

print("LM Finance Dictionary loaded.")
print(f"pypdf      : {PYPDF_AVAIL}")
print(f"pdfplumber : {PDFPLUMBER_AVAIL}")
print(f"Project    : {PROJECT_ROOT}")
print(f"Local PDFs : {LOCAL_PDF_DIR}")

LM Finance Dictionary loaded.
pypdf      : True
pdfplumber : True
Project    : C:\Users\rohit\Desktop\Rohit\RBI_Sentiment_Project
Local PDFs : C:\Users\rohit\Desktop\Rohit\RBI_Sentiment_Project\data\raw\pdfs


## Step 2 — Document link lists
### MPC Minutes (2010–2025)

In [23]:
MPC_DOCS = [
    ("Jul-2010", "https://rbidocs.rbi.org.in/rdocs/notification/PDFs/FQRJ270710.pdf"),
    ("Oct-2010", "https://rbidocs.rbi.org.in/rdocs/notification/PDFs/MSQRNO21110.pdf"),
    ("Jul-2011", "https://rbidocs.rbi.org.in/rdocs/Publications/PDFs/MMD250711FL.pdf"),
    ("Oct-2011", "https://rbidocs.rbi.org.in/rdocs/notification/PDFs/SQR251011FL.pdf"),
    ("Jul-2012", "https://rbidocs.rbi.org.in/rdocs/notification/PDFs/JUL1230FMP_12Q98AP.pdf"),
    ("Oct-2012", "https://rbidocs.rbi.org.in/rdocs/notification/PDFs/PSQ1030121694377F08.pdf"),
    ("Jul-2013", "https://rbidocs.rbi.org.in/rdocs/notification/PDFs/FS071301BC69D991.pdf"),
    ("Oct-2013", "https://rbidocs.rbi.org.in/rdocs/notification/PDFs/291013SQR61b74f346a.pdf"),
    ("Jun-2014", "https://rbidocs.rbi.org.in/rdocs/PressRelease/PDFs/IESB06201432A762B0B7.pdf"),
    ("Dec-2014", "https://rbidocs.rbi.org.in/rdocs/PressRelease/PDFs/IEPRFB6CBBBDB8FDFL.pdf"),
    ("Jun-2015", "https://rbidocs.rbi.org.in/rdocs/PressRelease/PDFs/SBIPO3CD71345E083438293499AB5A44AFC0A.PDF"),
    ("Dec-2015", "https://rbidocs.rbi.org.in/rdocs/PressRelease/PDFs/1278YSF263A51ECC9AC4D21BF547C48FF433B1A.PDF"),
    ("Jun-2016", "https://rbidocs.rbi.org.in/rdocs/PressRelease/PDFs/PR2834341F3AAE216E4EA599EE6F8E24FBFDDC.PDF"),
    ("Dec-2016", "https://rbidocs.rbi.org.in/rdocs/PressRelease/PDFs/PR1606667E13ADA7E54D059FE6BC962A332AF9.PDF"),
    ("Jun-2017", "https://rbidocs.rbi.org.in/rdocs/PressRelease/PDFs/PR34434806EBADEBFD48718D2FB27A80ACA827.PDF"),
    ("Dec-2017", "https://rbidocs.rbi.org.in/rdocs/PressRelease/PDFs/PR16916A79D2EEB1CA49F2BC1605B48DE765FA.PDF"),
    ("Jun-2018", "https://rbidocs.rbi.org.in/rdocs/PressRelease/PDFs/PR3320556E3589C58C4978B8E24686472796DD.PDF"),
    ("Dec-2018", "https://rbidocs.rbi.org.in/rdocs/PressRelease/PDFs/PR141100CEF33A69E046F7917B6DF02B474566.PDF"),
    ("Jun-2019", "https://rbidocs.rbi.org.in/rdocs/PressRelease/PDFs/PR3001D99C9154FF7E47169DB3EE95D3508F80.PDF"),
    ("Dec-2019", "https://rbidocs.rbi.org.in/rdocs/PressRelease/PDFs/PR1463B076CEDDC886439CAD1CBB5E99F25BE1.PDF"),
    ("Jun-2020", "https://rbidocs.rbi.org.in/rdocs/PressRelease/PDFs/PR24591AE8154A8C00484D85E29CA51184761D.PDF"),
    ("Dec-2020", "https://rbidocs.rbi.org.in/rdocs/PressRelease/PDFs/PR804A43CCF8760484AF3969B7B64CD6A7909.PDF"),
    ("Jun-2021", "https://rbidocs.rbi.org.in/rdocs/PressRelease/PDFs/PR390MPC7FA57E119D64473AA201098BE139DF4F.PDF"),
    ("Dec-2021", "https://rbidocs.rbi.org.in/rdocs/PressRelease/PDFs/PR1402MPCCEAB6A72D2544A98A02694F772FCC5A4.PDF"),
    ("Jun-2022", "https://rbidocs.rbi.org.in/rdocs/PressRelease/PDFs/PR406MPC90EFD348284F42B1B28AEB8C7B092394.PDF"),
    ("Dec-2022", "https://rbidocs.rbi.org.in/rdocs/PressRelease/PDFs/PR1420MPCF099880C30FD4CC38BAC999DF8C8B0FF.PDF"),
    ("Jun-2023", "https://rbidocs.rbi.org.in/rdocs/PressRelease/PDFs/PR449MINUTESMPC0878FC98204D417C85943F174D1A1A7A.PDF"),
    ("Dec-2023", "https://rbidocs.rbi.org.in/rdocs/PressRelease/PDFs/PR1531MPC028F84F6CF55474FAD12AF5AB0271F8C.PDF"),
    ("Jun-2024", "https://rbidocs.rbi.org.in/rdocs/PressRelease/PDFs/PR539MINUTESMPCD343876F4C594ED3834EA3A2C0B503B3.PDF"),
    ("Dec-2024", "https://rbidocs.rbi.org.in/rdocs/PressRelease/PDFs/PR174819B9C61C065E45FCA4F80C8DF099B1B8.PDF"),
    ("Jun-2025", "https://rbidocs.rbi.org.in/rdocs/PressRelease/PDFs/PR5704BFF4DD8A76B46498C92D3E3999DF6AA.PDF"),
]
print(f"MPC documents defined: {len(MPC_DOCS)}")

MPC documents defined: 31


### FSR Reports (2010–2025)

In [24]:
FSR_DOCS = [
    ("Mar-2010", "https://rbidocs.rbi.org.in/rdocs/PublicationReport/Pdfs/IFSR250310F.pdf"),
    ("Dec-2010", "https://rbidocs.rbi.org.in/rdocs//PublicationReport/Pdfs/FSR301210F.pdf"),
    ("Jun-2011", "https://rbidocs.rbi.org.in/rdocs/PublicationReport/Pdfs/FSR140611FL.pdf"),
    ("Dec-2011", "https://rbidocs.rbi.org.in/rdocs//PublicationReport/Pdfs/FSRD22122011_F.pdf"),
    ("Jun-2012", "https://rbidocs.rbi.org.in/rdocs//PublicationReport/Pdfs/FFSR260612_FL.pdf"),
    ("Dec-2012", "https://rbidocs.rbi.org.in/rdocs/PublicationReport/Pdfs/FFSR261212_FL.pdf"),
    ("Jun-2013", "https://rbidocs.rbi.org.in/rdocs//PublicationReport/Pdfs/FSPI260613FL.pdf"),
    ("Dec-2013", "https://rbidocs.rbi.org.in/rdocs//PublicationReport/Pdfs/FSRDEC301213_FL.pdf"),
    ("Jun-2014", "https://rbidocs.rbi.org.in/rdocs/PublicationReport/Pdfs/0FSR26062014F.pdf"),
    ("Dec-2014", "https://rbidocs.rbi.org.in/rdocs//PublicationReport/Pdfs/FSR29122014_FL.PDF"),
    ("Jun-2015", "https://rbidocs.rbi.org.in/rdocs//PublicationReport/Pdfs/0FS15A56030B88BD047B4A7124BA5AF1D8CF2.PDF"),
    ("Dec-2015", "https://rbidocs.rbi.org.in/rdocs/PublicationReport/Pdfs/0FSR6F7E7BC6C14F42E99568A80D9FF7BBA6.PDF"),
    ("Jun-2016", "https://rbidocs.rbi.org.in/rdocs/PublicationReport/Pdfs/0FSR2316BB76DB39BF964542B9D1EBE2CBC273E7.PDF"),
    ("Dec-2016", "https://rbidocs.rbi.org.in/rdocs/PublicationReport/Pdfs/0FSR_166BABD6ABE04B48AFB534749A1BF38882.PDF"),
    ("Jun-2017", "https://rbidocs.rbi.org.in/rdocs/PublicationReport/Pdfs/0FSR_30061794092D8D036447928A4B45880863B33E.PDF"),
    ("Dec-2017", "https://rbidocs.rbi.org.in/rdocs//PublicationReport/Pdfs/0FSR201730210986ADDA44E2A946A3F6C4408581.PDF"),
    ("Jun-2018", "https://rbidocs.rbi.org.in/rdocs/PublicationReport/Pdfs/0FSR_JUNE2018A3526EF7DC8640539C1420D256A470FC.PDF"),
    ("Dec-2018", "https://rbidocs.rbi.org.in/rdocs/PublicationReport/Pdfs/0FSRDECEMBER2018DAFEDD89C01C432786925639A4864F96.PDF"),
    ("Jun-2019", "https://rbidocs.rbi.org.in/rdocs/PublicationReport/Pdfs/FSRJUNE2019E5ECDDAD7E514756AFEF1E71CB2ADA2B.PDF"),
    ("Dec-2019", "https://rbidocs.rbi.org.in/rdocs/PublicationReport/Pdfs/0FSRDECEMBER20198C840246658946159CB3B94E8516F2EC.PDF"),
    ("Jun-2020", "https://rbidocs.rbi.org.in/rdocs/PublicationReport/Pdfs/0FSRJULY2020C084CED43CD1447D80B4789F7E49E499.PDF"),
    ("Jan-2021", "https://rbidocs.rbi.org.in/rdocs//PublicationReport/Pdfs/FSR_F06B552BF8B144B80B4AEFEDEB3D62218.PDF"),
    ("Jun-2021", "https://rbidocs.rbi.org.in/rdocs/PublicationReport/Pdfs/FSRJULY20210595CD3BEDFA466EBE9169BCE426E32C.PDF"),
    ("Dec-2021", "https://rbidocs.rbi.org.in/rdocs/PublicationReport/Pdfs/FSRDEC2021_FULL2D99E6548CD0478CA90EE717F2B85D45.PDF"),
    ("Jun-2022", "https://rbidocs.rbi.org.in/rdocs//PublicationReport/Pdfs/0FSRJUNE2022F758BFB27A9145A385FE9AC8D204AC82.PDF"),
    ("Dec-2022", "https://rbidocs.rbi.org.in/rdocs/PublicationReport/Pdfs/0FSRDECEMBER2022F93A2F188A394ACDB2FDDC2FCC0D07F0.PDF"),
    ("Jun-2023", "https://rbidocs.rbi.org.in/rdocs//PublicationReport/Pdfs/0FSRJUNE20231159B36F45EA406E9D704BBC8F73D785.PDF"),
    ("Dec-2023", "https://rbidocs.rbi.org.in/rdocs//PublicationReport/Pdfs/0FSRDECB815B9437D6D428F81D45C22BBF6C62A.PDF"),
    ("Jun-2024", "https://rbidocs.rbi.org.in/rdocs//PublicationReport/Pdfs/0FSRJUN2024_270620242B95CB128D1847A3ACAB5B5A4BEBF0DF.PDF"),
    ("Dec-2024", "https://rbidocs.rbi.org.in/rdocs//PublicationReport/Pdfs/FSR30122024F992B788790C44DCFBA4E8C9F98D912D9.PDF"),
    ("Jun-2025", "https://rbidocs.rbi.org.in/rdocs/PublicationReport/Pdfs/0FSRJUNE20253006258AE798B4484642AD861CC35BC2CB3D8E.PDF"),
]
print(f"FSR documents defined: {len(FSR_DOCS)}")

FSR documents defined: 31


## Step 3 — Core functions

### About the PDF download block
RBI's website returns **403 Forbidden** when requests come from cloud/server IP addresses.
This is a common government website restriction.

**On your own laptop** (home WiFi or college network) this usually works fine.
If it still blocks, use the manual download fallback below.

In [25]:
def extract_text_from_bytes(pdf_bytes):
    """
    Extract text from raw PDF bytes.
    Tries 3 methods in order — returns first successful result.
    """
    # ── Guard: check it is actually a PDF ────────────────────────────────────
    if not pdf_bytes or len(pdf_bytes) < 100:
        return ""
    if pdf_bytes[:4] != b'%PDF':
        # Server returned HTML/error page instead of PDF
        preview = pdf_bytes[:50].decode('utf-8','ignore')
        if '<!DOC' in preview or '<html' in preview.lower():
            print("  Blocked: server returned an HTML page, not a PDF.")
            print("  This usually means RBI is blocking automated downloads.")
            print("  -> Use manual download (Layer 2) or existing data (Layer 3).")
        else:
            print(f"  Not a PDF file. Starts with: {pdf_bytes[:20]}")
        return ""

    # ── Method 1: PyPDF2 with strict=False ───────────────────────────────────
    try:
        reader = PyPDF2.PdfReader(BytesIO(pdf_bytes), strict=False)
        text = "\n".join(p.extract_text() or "" for p in reader.pages)
        if text.strip():
            return text.strip()
    except Exception:
        pass

    # ── Method 2: pypdf ──────────────────────────────────────────────────────
    if PYPDF_AVAIL:
        try:
            reader2 = pypdf_Reader(BytesIO(pdf_bytes))
            text = "\n".join(p.extract_text() or "" for p in reader2.pages)
            if text.strip():
                return text.strip()
        except Exception:
            pass

    # ── Method 3: pdfplumber ─────────────────────────────────────────────────
    if PDFPLUMBER_AVAIL:
        try:
            with pdfplumber.open(BytesIO(pdf_bytes)) as pdf:
                text = "\n".join(p.extract_text() or "" for p in pdf.pages)
            if text.strip():
                return text.strip()
        except Exception:
            pass

    return ""


def extract_text_from_pdf_url(url, timeout=60):
    """Layer 1: Download PDF from URL then extract text."""
    try:
        headers = {
            'User-Agent'     : 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
            'Accept'         : 'application/pdf,application/octet-stream,*/*',
            'Accept-Encoding': 'identity',
            'Referer'        : 'https://rbidocs.rbi.org.in/',
        }
        r = requests.get(url, headers=headers, timeout=timeout,
                         verify=False, allow_redirects=True)
        if r.status_code == 403:
            print(f"  HTTP 403 Forbidden — RBI is blocking this IP/environment.")
            return ""
        r.raise_for_status()
        return extract_text_from_bytes(r.content)
    except requests.exceptions.Timeout:
        print(f"  Timeout ({timeout}s)")
        return ""
    except Exception as e:
        print(f"  Download error: {e}")
        return ""


def extract_text_from_local_pdf(date_label, source_label, local_dir):
    """
    Layer 2: Read PDF from local folder.
    File naming: MPC_Jul-2010.pdf or FSR_Mar-2010.pdf
    """
    filename = f"{source_label}_{date_label}.pdf"
    filepath = os.path.join(local_dir, filename)
    if not os.path.exists(filepath):
        return ""
    try:
        with open(filepath, 'rb') as f:
            pdf_bytes = f.read()
        return extract_text_from_bytes(pdf_bytes)
    except Exception as e:
        print(f"  Local file error: {e}")
        return ""


def compute_lm_sentiment(text):
    """
    Score text with Loughran-McDonald Finance Dictionary.
    Formula: Sentiment Index = (Positive - Negative) / Total Words
    """
    if not text or not text.strip():
        return {'Positive':0,'Negative':0,'Total_Words':0,
                'Sentiment_Index':0.0,'Polarity':0.0}
    tokens = lm_model.tokenize(text)
    score  = lm_model.get_score(tokens)
    total  = len(tokens)
    pos    = int(score['Positive'])
    neg    = int(score['Negative'])
    si     = (pos - neg) / total if total > 0 else 0.0
    return {
        'Positive'       : pos,
        'Negative'       : neg,
        'Total_Words'    : total,
        'Sentiment_Index': round(si, 6),
        'Polarity'       : round(float(score['Polarity']), 6),
    }

print("All functions defined.")

All functions defined.


## Step 4 — Quick connection test
Run this to check which layer will work for you.

In [26]:
import urllib3
urllib3.disable_warnings()

TEST_URL = "https://rbidocs.rbi.org.in/rdocs/PressRelease/PDFs/PR1531MPC028F84F6CF55474FAD12AF5AB0271F8C.PDF"

print("Testing Layer 1 (direct URL download)...")
try:
    r = requests.get(TEST_URL, timeout=20, verify=False,
                     headers={'User-Agent':'Mozilla/5.0','Accept':'application/pdf,*/*'})
    status = r.status_code
    is_pdf = r.content[:4] == b'%PDF'
    print(f"  Status  : {status}")
    print(f"  Is PDF  : {is_pdf}")
    if is_pdf:
        print("  LAYER 1 WORKS — direct download is available.")
        print("  Run Step 5 normally.")
    elif status == 403:
        print("  LAYER 1 BLOCKED (403) — RBI is blocking this network/IP.")
        print("\n  OPTIONS:")
        print("  Option A — Try from home WiFi/college network (usually works)")
        print("  Option B — Manually download PDFs and use Layer 2 (see instructions below)")
        print("  Option C — Use your existing scraped data with Layer 3 (fastest)")
    else:
        print(f"  Unexpected response. First bytes: {r.content[:30]}")
except Exception as e:
    print(f"  Connection failed: {e}")
    print("  Check your internet connection.")

print()
# Check Layer 2
local_pdfs = [f for f in os.listdir(LOCAL_PDF_DIR) if f.endswith('.pdf')]
print(f"Layer 2 (local PDFs): {len(local_pdfs)} PDF files in {LOCAL_PDF_DIR}")
if local_pdfs:
    print(f"  Files found: {local_pdfs[:5]}")

# Check Layer 3
old_mpc = os.path.join(DATA_PROC, 'mpc_sentiment.csv')
old_fsr = os.path.join(DATA_PROC, 'fsr_sentiment.csv')
has_old_mpc = os.path.exists(old_mpc)
has_old_fsr = os.path.exists(old_fsr)
print(f"Layer 3 (existing CSV): mpc_sentiment.csv={has_old_mpc}, fsr_sentiment.csv={has_old_fsr}")

Testing Layer 1 (direct URL download)...
  Status  : 200
  Is PDF  : False
  Unexpected response. First bytes: b'<!DOCTYPE html>\r\n<html><head>\r'

Layer 2 (local PDFs): 0 PDF files in C:\Users\rohit\Desktop\Rohit\RBI_Sentiment_Project\data\raw\pdfs
Layer 3 (existing CSV): mpc_sentiment.csv=True, fsr_sentiment.csv=True


## Step 5 — Manual download guide (Layer 2)
**Only needed if Layer 1 is blocked (403 error)**

If direct download is blocked, save PDFs manually in 10 minutes:

### Instructions
1. Create the folder: `data/raw/pdfs/` inside your project
2. Open each URL in your browser — it will download the PDF automatically
3. **Rename each file exactly** as shown below (e.g. `MPC_Jul-2010.pdf`)
4. Put all files in `data/raw/pdfs/`
5. Re-run Step 6 — it will pick them up automatically

### Quick download script — paste in browser console
```javascript
// Paste this in browser console to auto-click all MPC links
// Works on Chrome/Firefox
const urls = [
  "https://rbidocs.rbi.org.in/rdocs/notification/PDFs/FQRJ270710.pdf",
  // ... paste all URLs here
];
urls.forEach((url,i) => { setTimeout(()=>{ window.open(url,'_blank'); }, i*2000); });
```

### File naming format
```
MPC_Jul-2010.pdf    MPC_Oct-2010.pdf   ...  MPC_Jun-2025.pdf
FSR_Mar-2010.pdf    FSR_Dec-2010.pdf   ...  FSR_Jun-2025.pdf
```

## Step 6 — Run extraction (all 3 layers)
Automatically tries Layer 1 → Layer 2 → Layer 3 in order.

In [27]:
import urllib3
urllib3.disable_warnings()

MPC_PATH = os.path.join(DATA_PROC, 'mpc_sentiment_lm.csv')
FSR_PATH = os.path.join(DATA_PROC, 'fsr_sentiment_lm.csv')

# ── Check if valid results already exist ────────────────────────────────────
def is_valid_csv(path, min_ok=0.8):
    if not os.path.exists(path):
        return False
    try:
        df = pd.read_csv(path)
        return 'Total_Words' in df.columns and (df['Total_Words']>0).mean() >= min_ok
    except Exception:
        return False

if is_valid_csv(MPC_PATH) and is_valid_csv(FSR_PATH):
    mpc_df = pd.read_csv(MPC_PATH)
    fsr_df = pd.read_csv(FSR_PATH)
    mpc_ok = (mpc_df['Total_Words']>0).sum()
    fsr_ok = (fsr_df['Total_Words']>0).sum()
    print(f"Valid results found — loaded from CSV")
    print(f"  MPC: {mpc_ok}/{len(mpc_df)} OK | FSR: {fsr_ok}/{len(fsr_df)} OK")
    print("Delete the CSV files to force re-extraction.")

else:
    # Clean stale zero-filled files
    for p in [MPC_PATH, FSR_PATH]:
        if os.path.exists(p) and not is_valid_csv(p):
            os.remove(p)
            print(f"Removed stale file: {os.path.basename(p)}")

    # ── Single extraction function with 3-layer fallback ────────────────────
    def extract_one(date, url, source, local_dir):
        """Try URL first, then local file, returns (text, layer_used)."""

        # Layer 1: direct URL
        text = extract_text_from_pdf_url(url)
        if text:
            return text, "URL"

        # Layer 2: local PDF folder
        text = extract_text_from_local_pdf(date, source, local_dir)
        if text:
            return text, "LOCAL"

        return "", "FAILED"

    def run_extraction(doc_list, source_label):
        results = []
        n_ok = n_failed = 0
        layer_counts = {"URL":0, "LOCAL":0, "FAILED":0}

        for i, (date, url) in enumerate(doc_list):
            print(f"  [{i+1:02d}/{len(doc_list)}] {source_label} {date} ...", end="  ", flush=True)
            text, layer = extract_one(date, url, source_label, LOCAL_PDF_DIR)
            scores = compute_lm_sentiment(text)
            results.append({'Date':date, 'Source':source_label,
                            'Layer':layer, **scores, 'URL':url})
            layer_counts[layer] += 1
            if text:
                n_ok += 1
                print(f"{layer:<6}  Words={scores['Total_Words']:>6,}  SI={scores['Sentiment_Index']:+.5f}")
            else:
                n_failed += 1
                print(f"FAILED")

        print(f"\n  Result: {n_ok} OK, {n_failed} failed")
        print(f"  Layers used: URL={layer_counts['URL']} LOCAL={layer_counts['LOCAL']} FAILED={layer_counts['FAILED']}")
        return pd.DataFrame(results), n_ok

    print("="*65)
    print("Extracting MPC Minutes...")
    mpc_df, mpc_ok = run_extraction(MPC_DOCS, 'MPC')

    print("\nExtracting FSR Reports...")
    fsr_df, fsr_ok = run_extraction(FSR_DOCS, 'FSR')

    # ── Save only if enough succeeded ───────────────────────────────────────
    mpc_rate = mpc_ok / len(MPC_DOCS)
    fsr_rate = fsr_ok / len(FSR_DOCS)
    print("\n" + "="*65)

    if mpc_rate >= 0.8 and fsr_rate >= 0.8:
        mpc_df.to_csv(MPC_PATH, index=False)
        fsr_df.to_csv(FSR_PATH, index=False)
        print(f"SAVED: MPC={mpc_ok}/31 OK, FSR={fsr_ok}/31 OK")
    else:
        print(f"Too many failures — NOT saved (MPC={mpc_rate:.0%}, FSR={fsr_rate:.0%})")
        print("-> Proceeding to Layer 3 (use existing CSV data)")

Extracting MPC Minutes...
  [01/31] MPC Jul-2010 ...    Download error: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
FAILED
  [02/31] MPC Oct-2010 ...    Download error: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
FAILED
  [03/31] MPC Jul-2011 ...    Download error: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
FAILED
  [04/31] MPC Oct-2011 ...    Download error: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
FAILED
  [05/31] MPC Jul-2012 ...    Download error: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
FAILED
  [06/31] MPC Oct-2012 ...    Download error: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
FAILED
  [07/31] MPC Jul-2013 ...    Download error: ('Connection aborted.', RemoteDisconnected('Remote e

## Step 7 — Layer 3: Use existing scraped data
**Run this only if Steps 6 failed (Layer 1 and 2 both blocked).**

Your original project already has valid sentiment scores from a previous
successful scrape. This cell converts that data to our format so you can
continue to notebooks 03–06 without re-scraping.

In [28]:
# ── Layer 3: convert existing scraped CSVs ──────────────────────────────────
# Only runs if Layer 1/2 produced zero valid rows

def check_current_ok():
    """True if mpc_df and fsr_df already exist and have data."""
    try:
        return (mpc_df['Total_Words']>0).sum() >= 20
    except Exception:
        return False

if check_current_ok():
    print("Layer 1/2 already succeeded — Layer 3 not needed.")
    print(f"  MPC: {(mpc_df['Total_Words']>0).sum()} docs with data")
    print(f"  FSR: {(fsr_df['Total_Words']>0).sum()} docs with data")

else:
    print("Layer 1 and 2 both failed — using existing scraped data (Layer 3).")
    print()

    # Possible locations for old data
    old_candidates = {
        'mpc': [
            os.path.join(DATA_PROC, 'mpc_sentiment.csv'),
            os.path.join(PROJECT_ROOT, 'output', 'mpc_sentiment.csv'),
        ],
        'fsr': [
            os.path.join(DATA_PROC, 'fsr_sentiment.csv'),
            os.path.join(PROJECT_ROOT, 'output', 'fsr_sentiment.csv'),
        ]
    }

    def find_existing(candidates):
        for p in candidates:
            if os.path.exists(p):
                return p
        return None

    old_mpc_path = find_existing(old_candidates['mpc'])
    old_fsr_path = find_existing(old_candidates['fsr'])

    if not old_mpc_path or not old_fsr_path:
        print("ERROR: Could not find existing mpc_sentiment.csv or fsr_sentiment.csv")
        print("Please copy your old output CSVs into: data/processed/")
        print("Files needed:")
        print("  data/processed/mpc_sentiment.csv")
        print("  data/processed/fsr_sentiment.csv")
    else:
        old_mpc = pd.read_csv(old_mpc_path)
        old_fsr = pd.read_csv(old_fsr_path)
        print(f"Found existing data:")
        print(f"  MPC: {old_mpc_path} ({len(old_mpc)} rows)")
        print(f"  FSR: {old_fsr_path} ({len(old_fsr)} rows)")

        # ── Normalise column names ────────────────────────────────────────────
        col_map = {
            'Sentiment Index':'Sentiment_Index',
            'Positive Words' :'Positive',
            'Negative Words' :'Negative',
            'Total Words'    :'Total_Words',
        }
        old_mpc = old_mpc.rename(columns=col_map)
        old_fsr = old_fsr.rename(columns=col_map)

        # ── Add required columns ──────────────────────────────────────────────
        for df_old in [old_mpc, old_fsr]:
            if 'Polarity' not in df_old.columns:
                # Approximate polarity from sentiment index
                df_old['Polarity'] = df_old['Sentiment_Index'].apply(
                    lambda x: x / (abs(x)+1e-10) if x != 0 else 0.0
                ).round(6)
            if 'Layer' not in df_old.columns:
                df_old['Layer'] = 'EXISTING_CSV'
            if 'URL' not in df_old.columns:
                df_old['URL'] = ''

        # Ensure correct column order
        keep_cols = ['Date','Source','Positive','Negative',
                     'Total_Words','Sentiment_Index','Polarity','Layer','URL']
        mpc_df = old_mpc[[c for c in keep_cols if c in old_mpc.columns]]
        fsr_df = old_fsr[[c for c in keep_cols if c in old_fsr.columns]]

        mpc_df.to_csv(MPC_PATH, index=False)
        fsr_df.to_csv(FSR_PATH, index=False)

        print()
        print("Converted and saved successfully:")
        print(f"  mpc_sentiment_lm.csv — {len(mpc_df)} rows")
        print(f"  fsr_sentiment_lm.csv — {len(fsr_df)} rows")
        print()
        print("NOTE: These scores use your original custom dictionary (295 words).")
        print("If you later successfully run Layer 1 on your laptop, re-run this")
        print("notebook to get LM Finance Dictionary scores for better accuracy.")

Layer 1 and 2 both failed — using existing scraped data (Layer 3).

Found existing data:
  MPC: C:\Users\rohit\Desktop\Rohit\RBI_Sentiment_Project\data\processed\mpc_sentiment.csv (31 rows)
  FSR: C:\Users\rohit\Desktop\Rohit\RBI_Sentiment_Project\data\processed\fsr_sentiment.csv (31 rows)

Converted and saved successfully:
  mpc_sentiment_lm.csv — 31 rows
  fsr_sentiment_lm.csv — 31 rows

NOTE: These scores use your original custom dictionary (295 words).
If you later successfully run Layer 1 on your laptop, re-run this
notebook to get LM Finance Dictionary scores for better accuracy.


## Step 8 — Preview results

In [29]:
print("MPC SENTIMENT")
print("="*70)
show_cols = [c for c in ['Date','Total_Words','Positive','Negative',
                          'Sentiment_Index','Layer'] if c in mpc_df.columns]
print(mpc_df[show_cols].to_string(index=False))

print("\nFSR SENTIMENT")
print("="*70)
show_cols_f = [c for c in ['Date','Total_Words','Positive','Negative',
                             'Sentiment_Index','Layer'] if c in fsr_df.columns]
print(fsr_df[show_cols_f].to_string(index=False))

print("\nSUMMARY")
print("="*70)
for label, df in [('MPC', mpc_df), ('FSR', fsr_df)]:
    ok = (df['Total_Words']>0).sum()
    if ok > 0:
        avg = df[df['Total_Words']>0]['Sentiment_Index'].mean()
        mn  = df[df['Total_Words']>0]['Sentiment_Index'].min()
        mx  = df[df['Total_Words']>0]['Sentiment_Index'].max()
        print(f"  {label}: {ok}/31 docs | SI avg={avg:+.5f} min={mn:+.5f} max={mx:+.5f}")

MPC SENTIMENT
  Date  Total_Words  Positive  Negative  Sentiment_Index        Layer
Jul-10         4964        40        33         0.001410 EXISTING_CSV
Oct-10        12395        63        49         0.001129 EXISTING_CSV
Jul-11        28214       101       170        -0.002446 EXISTING_CSV
Oct-11        12018        58        64        -0.000499 EXISTING_CSV
Jul-12         4375        15        63        -0.010971 EXISTING_CSV
Oct-12        11725        51        85        -0.002900 EXISTING_CSV
Jul-13         3890        19        42        -0.005913 EXISTING_CSV
Oct-13         3845        21        18         0.000780 EXISTING_CSV
Jun-14         1749        13        13         0.000000 EXISTING_CSV
Dec-14         2296        17        26        -0.003920 EXISTING_CSV
Jun-15         2362        15        37        -0.009314 EXISTING_CSV
Dec-15         2009        15        20        -0.002489 EXISTING_CSV
Jun-16         2262        13        14        -0.000442 EXISTING_CSV
Dec-16

## Step 9 — Combine into master sentiment CSV

In [30]:
mpc_ok = (mpc_df['Total_Words']>0).sum()
fsr_ok = (fsr_df['Total_Words']>0).sum()

if mpc_ok == 0 or fsr_ok == 0:
    print("ERROR: One or both datasets are empty. Fix extraction before combining.")
else:
    combined = pd.DataFrame({
        'Period'         : mpc_df['Date'].values,
        'MPC_Sentiment'  : mpc_df['Sentiment_Index'].values,
        'MPC_Polarity'   : mpc_df['Polarity'].values,
        'MPC_Positive'   : mpc_df['Positive'].values,
        'MPC_Negative'   : mpc_df['Negative'].values,
        'MPC_TotalWords' : mpc_df['Total_Words'].values,
        'FSR_Sentiment'  : fsr_df['Sentiment_Index'].values,
        'FSR_Polarity'   : fsr_df['Polarity'].values,
        'FSR_Positive'   : fsr_df['Positive'].values,
        'FSR_Negative'   : fsr_df['Negative'].values,
        'FSR_TotalWords' : fsr_df['Total_Words'].values,
    })
    out = os.path.join(DATA_PROC, 'combined_sentiment_lm.csv')
    combined.to_csv(out, index=False)
    print(f"Saved: combined_sentiment_lm.csv")
    print(f"Shape: {combined.shape[0]} rows x {combined.shape[1]} columns")
    print()
    print(combined[['Period','MPC_Sentiment','FSR_Sentiment']].to_string(index=False))

Saved: combined_sentiment_lm.csv
Shape: 31 rows x 11 columns

Period  MPC_Sentiment  FSR_Sentiment
Jul-10       0.001410      -0.003694
Oct-10       0.001129      -0.003915
Jul-11      -0.002446      -0.005365
Oct-11      -0.000499      -0.007830
Jul-12      -0.010971      -0.008530
Oct-12      -0.002900      -0.008051
Jul-13      -0.005913      -0.005977
Oct-13       0.000780      -0.007977
Jun-14       0.000000      -0.005361
Dec-14      -0.003920      -0.003993
Jun-15      -0.009314      -0.004150
Dec-15      -0.002489      -0.007503
Jun-16      -0.000442      -0.005397
Dec-16      -0.006456      -0.005058
Jun-17      -0.002428      -0.003734
Dec-17       0.000299      -0.004733
Jun-18       0.000871      -0.006149
Dec-18      -0.004338      -0.004963
Jun-19      -0.007646      -0.006375
Dec-19      -0.002924      -0.006486
Jun-20      -0.009206      -0.007447
Dec-20       0.004220      -0.005678
Jun-21       0.001740      -0.004602
Dec-21       0.000689      -0.005188
Jun-22      -

In [31]:
print("="*55)
print("NOTEBOOK 02 COMPLETE")
print("="*55)
for fname, fpath in [
    ('mpc_sentiment_lm.csv',  MPC_PATH),
    ('fsr_sentiment_lm.csv',  FSR_PATH),
    ('combined_sentiment_lm.csv', os.path.join(DATA_PROC,'combined_sentiment_lm.csv')),
]:
    if os.path.exists(fpath):
        kb = os.path.getsize(fpath)//1024
        print(f"  OK      {fname}  ({kb} KB)")
    else:
        print(f"  MISSING {fname}")
print("\nProceed to Notebook 03 — Stability Index.")

NOTEBOOK 02 COMPLETE
  OK      mpc_sentiment_lm.csv  (1 KB)
  OK      fsr_sentiment_lm.csv  (2 KB)
  OK      combined_sentiment_lm.csv  (2 KB)

Proceed to Notebook 03 — Stability Index.
